# Yield Curve Construction from Bond Prices
# Bootstrapping Zero-Coupon Rates for Fixed Income Analysis

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## 1. Data Preparation

We will define a small set of hypothetical bonds with different maturities, coupon rates, and prices.


In [ ]:
import pandas as pd

# Hypothetical bond data: Price, Maturity (years), Coupon Rate (annual)
bond_data = pd.DataFrame({
    'Maturity': [0.5, 1.0, 2.0, 3.0],
    'Coupon':   [0.00, 0.03, 0.04, 0.05],  # annual coupon as fraction of face
    'Price':    [98.5, 101.0, 102.0, 103.5]
})
bond_data

## 3. Bootstrapping Discount Factors

Use the observed prices and previously derived discount factors to solve for unknowns sequentially.


In [ ]:
# Sort bonds by maturity
bond_data = bond_data.sort_values('Maturity').reset_index(drop=True)

discount_factors = {}
results = []

for _, row in bond_data.iterrows():
    M = row['Maturity']                     # in years
    C = row['Coupon'] * 100                 # coupon payment per 100 face
    P = row['Price']                        # market price per 100 face
    F = 100                                 # face value

    if M == 0.5:
        # Zero-coupon bond at 6 months
        df = P / F
    else:
        # Annual coupons at each integer year
        # Sum PV of coupon payments up to maturity - 1
        sum_pv = sum(C * discount_factors[t] for t in discount_factors if t < M)
        # Price = PV(coupons) + PV(final coupon + principal)
        df = (P - sum_pv) / (F + C)

    discount_factors[M] = df
    # Compute annual zero rate
    zero_rate = (1 / df)**(1 / M) - 1
    results.append({'Maturity': M, 'Discount Factor': df, 'Zero Rate': zero_rate})

df_table = pd.DataFrame(results)
df_table


## 4. Plotting the Bootstrapped Yield Curve

Visualize annual zero rates against maturity.


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(df_table['Maturity'], df_table['Zero Rate'], marker='o')
plt.title('Bootstrapped Zero-Coupon Yield Curve')
plt.xlabel('Maturity (Years)')
plt.ylabel('Zero Rate (Annual)')
plt.grid(True)
plt.show()

5. Interpolating for a Smooth Curve

Use cubic interpolation to estimate rates at intermediate maturities.


In [ ]:
from scipy.interpolate import interp1d
import numpy as np

interp_func = interp1d(df_table['Maturity'],
                       df_table['Zero Rate'], kind='cubic')
maturities_smooth = np.linspace(
    df_table['Maturity'].min(), df_table['Maturity'].max(), 100)
rates_smooth = interp_func(maturities_smooth)

plt.figure()
plt.plot(maturities_smooth, rates_smooth)
plt.title('Interpolated Zero-Coupon Yield Curve (Cubic)')
plt.xlabel('Maturity (Years)')
plt.ylabel('Zero Rate')
plt.grid(True)
plt.show()

## Conclusion

In this notebook, we demonstrated how to bootstrap discount factors and derive annual zero-coupon rates from observed bond prices with various maturities and coupons. We then visualized the resulting yield curve and applied interpolation for a smooth representation. This process is fundamental in fixed income analysis for pricing interest rate derivatives, risk management, and understanding the term structure of interest rates.
